In [ ]:
# Client OMD

import os

import requests
from dotenv import load_dotenv

# override=True : sinon une valeur deja chargee dans le kernel n'est jamais remplacee
load_dotenv(override=True)


def _api_base(raw: str) -> str:
    """Normalise OMD_HOST_PORT : le SDK OMD veut le suffixe /api, pas le notebook."""
    base = raw.rstrip("/")
    return base if base.endswith("/api") else f"{base}/api"


OM_HOST = _api_base(os.environ["OMD_HOST_PORT"])
JWT_TOKEN = os.environ["OMD_JWT_TOKEN"]

headers = {
    "Authorization": f"Bearer {JWT_TOKEN}",
    "Content-Type": "application/json"
}


In [14]:
def search_metadata(query: str, index: str = "table_search_index", size: int = 10):
    """
    index peut etre: table_search_index, topic_search_index, dashboard_search_index,
    pipeline_search_index, mlmodel_search_index, all (tous types confondus), etc.
    """
    params = {
        "q": query,
        "index": index,
        "from": 0,
        "size": size,
        "sort_field": "_score",
        "sort_order": "desc"
    }
    resp = requests.get(f"{OM_HOST}/v1/search/query", headers=headers, params=params)
    resp.raise_for_status()
    if "application/json" not in resp.headers.get("Content-Type", ""):
        # OMD sert l'index.html de l'UI en 200 sur une route inconnue
        raise RuntimeError(f"reponse non-JSON depuis {resp.url} — verifie OMD_HOST_PORT")
    return resp.json()


print("base API:", OM_HOST)
results = search_metadata("clients", index="table_search_index")

for hit in results["hits"]["hits"]:
    src = hit["_source"]
    print(f"{src['fullyQualifiedName']} — {src.get('description') or 'no description'} (score: {hit['_score']})")


base API: http://172.16.240.10:8585/api
banking db.default.dvsys.dba_dv_simulation_log — no description (score: 2.6198206)
banking db.default.audsys.AUD$UNIFIED — no description (score: 1.9557474)
banking db.default.audsys.cdb_unified_audit_trail — All audit trail entries in all containers (score: 1.9557474)
banking db.default.audsys.unified_audit_trail — All audit trail entries (score: 1.9557474)
